# 22 · Single-run decision experiment: conventional access and home-origin transitions

**Questions.** (M10) How much does reselecting purpose targets inside a reduced paired household bootstrap change the contrast intervals? (M7) Does the observed metropolitan FPT gap differ from a conventional travel-time accessibility benchmark? (M1) Does the FPT contrast remain if empirical sex-specific transition rows use only trips starting in their household's home UTAM? Already completed M2–M4 and M8 are not repeated here.

The fixed-time benchmark uses survey-weighted mean positive reported duration on each observed OD edge, Dijkstra shortest path, identical scope-specific validated core-50 destination UTAMs, and destination weights proportional to pooled supported arrivals. It reports weighted opportunity share accessible within 45 and 60 minutes. It is a revealed-flow *proxy*, not a complete multimodal routing graph or census of real jobs. Residence distributions, time surface and targets are fixed within each scope; the pooled time surface has no sex-specific behavioral transitions. Sex differences in this benchmark are therefore residential, by construction. Do not infer causal or superiority claims from a difference between metrics.

The home-origin check fixes the population reference, SCC, targets and residential distributions; only empirical sex-specific rows use home-origin trips. It is descriptive and does not inherit the primary bootstrap intervals. Shared Poisson household resampling gives uncertainty only for the conventional benchmark, conditional on the fixed time surface and targets. Run once, inspect the decision card, then edit the paper according to the outcome.


In [ ]:
%pip -q install pandas pyarrow numpy scipy matplotlib seaborn geopandas networkx


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import csv, json, os, re, shutil, unicodedata, zipfile

import geopandas as gpd
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns

ROOT = Path("/content/colombia-stochastic-access")
RAW = ROOT / "data/raw/bogota_em2023"
INTERIM = ROOT / "data/interim/bogota_em2023_gate23"
OUT = ROOT / "outputs/bogota_em2023/gate23"
DRIVE_BASE = Path("/content/drive/MyDrive/colombia-stochastic-access-data/bogota_em2023")
for directory in (RAW, INTERIM, OUT):
    directory.mkdir(parents=True, exist_ok=True)

SOURCE_URL = "https://observatorio.movilidadbogota.gov.co/sites/default/files/2025-09/05_Base%20datos%20procesada%20EODH.zip"
ZONING_URL = "https://observatorio.movilidadbogota.gov.co/sites/default/files/2025-09/03_Zonificacion%20EODH%20%281%29.zip"
PURPOSES = ("employment", "education", "health")
EFFECTS = ("female_minus_male_total", "residential_contribution", "kernel_contribution")
SEX_LEVELS = ("mujer", "hombre")
FLOW_THRESHOLDS = (0.50, 0.80)
TAU = 20.0
BOOTSTRAP_REPLICATES = 500
RANDOM_SEED = 202316
MIN_DESTINATION_SAMPLE = 5
MIN_DESTINATION_ESS = 3.0

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception as exc:
    print(f"Entorno fuera de Colab o Drive no montado: {exc}")

G4 = Path(os.environ.get("GATE4_DIR_OVERRIDE", DRIVE_BASE / "results/gate4"))
G6 = Path(os.environ.get("GATE6_DIR_OVERRIDE", DRIVE_BASE / "results/gate6"))
G11 = Path(os.environ.get("GATE11_DIR_OVERRIDE", DRIVE_BASE / "results/gate11"))
G12 = Path(os.environ.get("GATE12_DIR_OVERRIDE", DRIVE_BASE / "results/gate12"))
G13 = Path(os.environ.get("GATE13_DIR_OVERRIDE", DRIVE_BASE / "results/gate13"))
paths = {
    "region_edges": G4 / "transition_edges_utam.parquet",
    "region_targets": G6 / "purpose_target_sets.csv",
    "region_point": G11 / "final_female_male_point_estimates.csv",
    "city_point": G12 / "bogota_dc_internal_female_male_point_estimates.csv",
    "city_targets": G12 / "bogota_dc_internal_core50_targets.csv",
    "paired_point": G13 / "paired_scope_contrasts_point.csv",
    "gate11": G11 / "gate_11.json",
    "gate12": G12 / "gate_12.json",
    "gate13": G13 / "gate_13.json",
    "gate22": DRIVE_BASE / "results/gate22/gate_22.json",
}
missing = [str(path) for path in paths.values() if not path.exists()]
if missing:
    raise FileNotFoundError("Missing validated inputs: " + ", ".join(missing))
for number in (11, 12, 13, 22):
    gate = json.loads(paths[f"gate{number}"].read_text(encoding="utf-8"))
    if not bool(gate.get("gate", {}).get(f"gate_{number}_passed", False)):
        raise RuntimeError(f"Gate {number} failed")
print("Validated dependencies: Gates 11, 12, 13")
print("Output:", OUT)


In [ ]:
def valid_zip(path):
    path = Path(path)
    return path.exists() and path.stat().st_size > 0 and zipfile.is_zipfile(path)

def ensure_archive(runtime_name, drive_name, url, label):
    runtime = RAW / runtime_name
    cached = DRIVE_BASE / drive_name
    if valid_zip(runtime):
        return runtime, "runtime_cache"
    if valid_zip(cached):
        shutil.copy2(cached, runtime)
        return runtime, "google_drive_cache"
    from google.colab import files
    print(f"Descargue y seleccione el ZIP de {label}:\n{url}")
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Suba exactamente un ZIP")
    name, content = next(iter(uploaded.items()))
    temporary = runtime.with_suffix(".zip.upload")
    temporary.write_bytes(content)
    del content
    if not valid_zip(temporary):
        temporary.unlink(missing_ok=True)
        raise ValueError(f"{name} no es un ZIP válido")
    temporary.replace(runtime)
    DRIVE_BASE.mkdir(parents=True, exist_ok=True)
    shutil.copy2(runtime, cached)
    return runtime, "manual_upload"

def safe_extract_recursive(zip_path, destination):
    destination.mkdir(parents=True, exist_ok=True)
    queue = [(Path(zip_path), destination)]
    visited = set()
    while queue:
        current, target = queue.pop(0)
        signature = (str(current.resolve()), str(target.resolve()))
        if signature in visited:
            continue
        visited.add(signature)
        root = target.resolve()
        with zipfile.ZipFile(current) as zipped:
            for member in zipped.infolist():
                resolved = (target / member.filename).resolve()
                if root != resolved and root not in resolved.parents:
                    raise ValueError(f"Ruta insegura: {member.filename}")
            zipped.extractall(target)
        for nested in target.rglob("*.zip"):
            nested_target = nested.parent / nested.stem
            if (str(nested.resolve()), str(nested_target.resolve())) not in visited:
                queue.append((nested, nested_target))

survey_archive, survey_method = ensure_archive(
    "processed_edoh.zip", "processed_edoh.zip", SOURCE_URL, "microdatos"
)
zoning_archive, zoning_method = ensure_archive(
    "zoning_edoh.zip", "zoning_edoh.zip", ZONING_URL, "zonificación"
)
safe_extract_recursive(survey_archive, INTERIM / "processed_edoh")
safe_extract_recursive(zoning_archive, INTERIM / "zoning_edoh")
print("Microdatos:", survey_method, "| Zonificación:", zoning_method)


In [ ]:
def normalize(value):
    value = unicodedata.normalize("NFKD", str(value))
    value = "".join(char for char in value if not unicodedata.combining(char))
    return re.sub(r"[^a-z0-9]+", "_", value.lower().strip()).strip("_")

def clean_key(series):
    cleaned = series.astype("string").str.strip().str.replace(r"\.0$", "", regex=True)
    return cleaned.mask(cleaned.map(normalize).isin(
        ["", "nan", "none", "na", "no_aplica", "sin_informacion"]
    ))

def canonical_utam(series):
    cleaned = clean_key(series)
    def one(value):
        if pd.isna(value):
            return pd.NA
        text = str(value).strip().upper().replace(" ", "")
        match = re.fullmatch(r"UPR0*(\d+)", text)
        if match:
            return f"UPR{int(match.group(1)):04d}"
        match = re.fullmatch(r"(?:UTAM)?0*(\d+)", text)
        return f"UTAM{int(match.group(1)):03d}" if match else text
    return cleaned.map(one).astype("string")

def parse_number(series):
    raw = series.astype("string").str.strip()
    if float(raw.str.contains(",", regex=False, na=False).mean()) > 0.05:
        raw = raw.str.replace(".", "", regex=False).str.replace(",", ".", regex=False)
    return pd.to_numeric(raw, errors="coerce")

def csv_format(path):
    for encoding in ("utf-8-sig", "latin-1"):
        try:
            with path.open("r", encoding=encoding, errors="strict") as handle:
                sample = handle.read(32768)
            return encoding, csv.Sniffer().sniff(sample, delimiters=",;\t|").delimiter
        except (UnicodeDecodeError, csv.Error):
            continue
    raise ValueError(f"Formato no detectado: {path}")

def find_module(token, excluded=()):
    candidates = [
        path for path in (INTERIM / "processed_edoh").rglob("*.csv")
        if token in normalize(path.stem)
        and not any(item in normalize(path.stem) for item in excluded)
    ]
    if not candidates:
        raise FileNotFoundError(token)
    return sorted(candidates, key=lambda path: (len(path.parts), len(path.name)))[0]

def read_selected(path, required):
    encoding, delimiter = csv_format(path)
    header = pd.read_csv(path, sep=delimiter, encoding=encoding, nrows=0)
    mapping = {normalize(column): column for column in header.columns}
    absent = sorted(set(required) - set(mapping))
    if absent:
        raise KeyError(f"Faltan columnas {absent}")
    frame = pd.read_csv(
        path, sep=delimiter, encoding=encoding,
        usecols=[mapping[column] for column in required],
        dtype="string", low_memory=False,
    )
    frame.columns = [normalize(column) for column in frame.columns]
    return frame

trips = read_selected(
    find_module("viaje", excluded=("etapa",)),
    ["key_hg", "utam_ori", "utam_des", "fexp_vj", "sexo", "motivo_viaje"],
)
persons = read_selected(
    find_module("persona"),
    ["key_hg", "cod_utam_hg", "fexp_per_5anos", "sexo"],
)
trips["household"] = clean_key(trips["key_hg"])
trips["utam_ori"] = canonical_utam(trips["utam_ori"])
trips["utam_des"] = canonical_utam(trips["utam_des"])
trips["weight"] = parse_number(trips["fexp_vj"])
trips["sex_group"] = trips["sexo"].map(normalize)
trips["purpose_normalized"] = trips["motivo_viaje"].map(normalize)
persons["household"] = clean_key(persons["key_hg"])
persons["residence_utam"] = canonical_utam(persons["cod_utam_hg"])
persons["weight"] = parse_number(persons["fexp_per_5anos"])
persons["sex_group"] = persons["sexo"].map(normalize)
trips = trips[
    trips[["household", "utam_ori", "utam_des"]].notna().all(axis=1)
    & trips["weight"].gt(0)
].reset_index(drop=True)
persons = persons[
    persons[["household", "residence_utam"]].notna().all(axis=1)
    & persons["weight"].gt(0)
].reset_index(drop=True)

PURPOSE_PATTERNS = {
    "employment": r"^a_trabajar$",
    "education": r"estudi|educa|coleg|univers|jardin|escuela",
    "health": r"salud|medic|hospital|clinic|consulta|terapia",
    "job_search": r"buscar_trabajo",
    "mobile_work": r"conduzco_vehiculo_como_forma_de_trabajo",
}
def classify_purpose(value):
    matches = [
        purpose for purpose, pattern in PURPOSE_PATTERNS.items()
        if re.search(pattern, value)
    ]
    return "ambiguous" if len(matches) > 1 else (matches[0] if matches else "other")
trips["purpose_group"] = trips["purpose_normalized"].map(classify_purpose)

spatial_paths = (
    list((INTERIM / "zoning_edoh").rglob("*.shp"))
    + list((INTERIM / "zoning_edoh").rglob("*.geojson"))
    + list((INTERIM / "zoning_edoh").rglob("*.gpkg"))
)
candidates = []
for path in spatial_paths:
    try:
        columns = {normalize(column): column for column in gpd.read_file(path, rows=5).columns}
        if "utam" in columns:
            candidates.append(path)
    except Exception:
        continue
if not candidates:
    raise FileNotFoundError("Capa UTAM")
utam_path = sorted(
    candidates, key=lambda path: ("utam2023" not in normalize(path.stem), len(path.parts))
)[0]
layer = gpd.read_file(utam_path)
column_map = {normalize(column): column for column in layer.columns}
municipality_key = (
    "munnombre" if "munnombre" in column_map
    else next(key for key in column_map if "mun" in key and "nombre" in key)
)
lookup = pd.DataFrame({
    "utam": canonical_utam(layer[column_map["utam"]]),
    "municipality": layer[column_map[municipality_key]].astype("string").str.strip(),
}).dropna().drop_duplicates("utam")
lookup["municipality_normalized"] = lookup["municipality"].map(normalize)
print("Viajes:", len(trips), "| Personas:", len(persons), "| Capa:", utam_path)


In [ ]:
raw = read_selected(find_module("viaje", excluded=("etapa",)),
    ["key_hg", "utam_ori", "utam_des", "fexp_vj", "duracion_min"])
raw["household"] = clean_key(raw["key_hg"])
raw["utam_ori"] = canonical_utam(raw["utam_ori"])
raw["utam_des"] = canonical_utam(raw["utam_des"])
raw["weight"] = parse_number(raw["fexp_vj"])
raw = raw[raw[["household", "utam_ori", "utam_des"]].notna().all(axis=1)
          & raw["weight"].gt(0)].reset_index(drop=True)
key = ["household", "utam_ori", "utam_des", "weight"]
if not raw[key].equals(trips[key]):
    raise RuntimeError("Duration records do not reproduce validated trip selection")
trips["duration_min"] = parse_number(raw["duracion_min"])
residence_counts = persons.groupby("household")["residence_utam"].nunique()
home_map = persons[persons["household"].isin(residence_counts.index[residence_counts.eq(1)])
    ].drop_duplicates("household").set_index("household")["residence_utam"]
trips["home_utam"] = trips["household"].map(home_map)
trips["from_home"] = trips["utam_ori"].eq(trips["home_utam"]).fillna(False)
edges = pd.read_parquet(paths["region_edges"])
node_set = set(canonical_utam(edges["utam_ori"])) | set(canonical_utam(edges["utam_des"]))
region_trips = trips[trips["utam_ori"].isin(node_set) & trips["utam_des"].isin(node_set)].copy()
region_persons = persons[persons["residence_utam"].isin(node_set)].copy()
mun = lookup.set_index("utam")["municipality_normalized"].to_dict()
city_nodes = {u for u in node_set if "bogota" in str(mun.get(u, ""))}
if not city_nodes:
    raise RuntimeError("Missing Bogotá UTAMs")
support = {
    "regional_trip_rows": len(region_trips),
    "duration_weight_coverage": float(region_trips.loc[region_trips["duration_min"].gt(0), "weight"].sum()/region_trips["weight"].sum()),
    "home_origin_trip_weight_share": float(region_trips.loc[region_trips["from_home"], "weight"].sum()/region_trips["weight"].sum()),
    "households_with_multiple_residences": int(residence_counts.gt(1).sum()),
    "trip_rows_without_unique_home_utam": int(region_trips["home_utam"].isna().sum())
}
for label,candidates in (("district",city_nodes),("region",node_set)):
    eligible=region_trips[region_trips["utam_ori"].isin(candidates) &
                          region_trips["utam_des"].isin(candidates)]
    for sex in SEX_LEVELS:
        group=eligible[eligible["sex_group"].eq(sex).fillna(False)]
        home=group[group["from_home"]]
        support[f"{label}_{sex}_sampled_trips"]=int(len(group))
        support[f"{label}_{sex}_sampled_home_trips"]=int(len(home))
        support[f"{label}_{sex}_home_trip_weight_share"]=float(home["weight"].sum()/group["weight"].sum())
        w=home["weight"].to_numpy(dtype=float)
        support[f"{label}_{sex}_home_trip_ess"]=float(w.sum()**2/np.square(w).sum()) if len(w) else 0.0
display(pd.Series(support))


In [ ]:
def saved_targets(path):
    df = pd.read_csv(path, dtype={"utam": "string"})
    df["utam"] = canonical_utam(df["utam"])
    if "target_definition" in df.columns:
        df = df[df["target_definition"].eq("core_50")]
    return {p: set(df.loc[df["purpose_group"].eq(p), "utam"])
            for p in PURPOSES}

def model(scope, home_only=False):
    n = len(scope["nodes"])
    kernels = {}
    for sex in SEX_LEVELS:
        mask = scope["trips"]["sex_group"].eq(sex).fillna(False).to_numpy(dtype=bool)
        if home_only:
            mask &= scope["trips"]["from_home"].to_numpy(dtype=bool)
        origin = scope["origin"][mask]
        destination = scope["destination"][mask]
        weight = scope["trip_weight"][mask]
        W = np.zeros((n,n))
        np.add.at(W,(origin,destination),weight)
        out = W.sum(axis=1)
        sumsq = np.bincount(origin,weights=weight**2,minlength=n)
        ess = np.divide(out**2,sumsq,out=np.zeros(n),where=sumsq>0)
        empirical = np.divide(W,out[:,None],out=np.zeros_like(W),where=out[:,None]>0)
        omega = ess/(ess+TAU)
        kernels[sex] = omega[:,None]*empirical+(1-omega[:,None])*scope["P0"]
    records = []
    for p in PURPOSES:
        target = np.array(sorted(scope["index"][u] for u in scope["targets"][p]))
        outside = np.array([i for i in range(n) if i not in set(target)])
        values = {}
        for sex in SEX_LEVELS:
            Q = kernels[sex][np.ix_(outside,outside)]
            h = np.zeros(n)
            h[outside] = np.linalg.solve(np.eye(len(outside))-Q,np.ones(len(outside)))
            mu = scope["mu"][sex].copy()
            mu[target] = 0
            mu /= mu.sum()
            values[sex] = float(mu @ h)
        f,m = values["mujer"], values["hombre"]
        records.append({"scope":scope["name"],"purpose_group":p,"female_fpt":f,
                        "male_fpt":m,"gap":f-m,"relative_gap":(f-m)/((f+m)/2)})
    return pd.DataFrame(records)

def make_scope(name, candidates, target_path, point_path):
    t = region_trips[region_trips["utam_ori"].isin(candidates) &
                     region_trips["utam_des"].isin(candidates)].copy()
    graph = nx.DiGraph()
    summed = t.groupby(["utam_ori","utam_des"],observed=True)["weight"].sum()
    graph.add_weighted_edges_from((u,v,float(w)) for (u,v),w in summed.items())
    component = sorted(nx.strongly_connected_components(graph),key=len,reverse=True)[0]
    nodes = sorted(component)
    selected = t[t["utam_ori"].isin(component) & t["utam_des"].isin(component)].reset_index(drop=True)
    residents = region_persons[region_persons["residence_utam"].isin(component)].reset_index(drop=True)
    index = {u:i for i,u in enumerate(nodes)}
    ori = selected["utam_ori"].map(index).to_numpy(dtype=int)
    dst = selected["utam_des"].map(index).to_numpy(dtype=int)
    wt = selected["weight"].to_numpy(dtype=float)
    W0 = np.zeros((len(nodes),len(nodes)))
    np.add.at(W0,(ori,dst),wt)
    if np.any(W0.sum(axis=1)<=0):
        raise RuntimeError(f"{name}: zero population row")
    P0 = W0/W0.sum(axis=1,keepdims=True)
    targets = saved_targets(target_path)
    for p in PURPOSES:
        if not targets[p] or not targets[p] <= set(nodes):
            raise RuntimeError(f"{name}: missing {p} targets")
    mu = {}
    for sex in SEX_LEVELS:
        subset = residents[residents["sex_group"].eq(sex).fillna(False)]
        mass = np.bincount(subset["residence_utam"].map(index).to_numpy(dtype=int),
                           weights=subset["weight"].to_numpy(dtype=float),minlength=len(nodes))
        mu[sex] = mass/mass.sum()
    result = dict(name=name,nodes=nodes,index=index,trips=selected,residents=residents,
                  origin=ori,destination=dst,trip_weight=wt,P0=P0,targets=targets,mu=mu)
    computed = model(result)
    reference = pd.read_csv(point_path)
    check = computed.merge(reference,on="purpose_group",validate="one_to_one")
    error = float((check["gap"]-check["female_minus_male_total"]).abs().max())
    if error>1e-8:
        raise RuntimeError(f"{name}: primary FPT mismatch {error}")
    return result,error

region,region_error = make_scope("bogota_region",node_set,paths["region_targets"],paths["region_point"])
city,city_error = make_scope("bogota_dc",city_nodes,paths["city_targets"],paths["city_point"])
if region["nodes"] != sorted(node_set):
    raise RuntimeError("Regional baseline lost a validated UTAM")
print("FPT replication errors:",region_error,city_error)
print("Nodes:",len(region["nodes"]),len(city["nodes"]))


## Conventional pooled-time benchmark on the same purpose target sets


In [ ]:
from scipy.sparse.csgraph import shortest_path
def deterministic_surface(scope):
    t = scope["trips"]
    valid = t["duration_min"].gt(0).fillna(False)
    d = t.loc[valid,["utam_ori","utam_des","duration_min","weight"]].copy()
    d["cost_product"] = d["duration_min"]*d["weight"]
    edges_d = d.groupby(["utam_ori","utam_des"],observed=True).agg(
        weighted_minutes=("cost_product","sum"),covered_weight=("weight","sum"))
    edges_d["minutes"] = edges_d["weighted_minutes"]/edges_d["covered_weight"]
    total_edges = t.groupby(["utam_ori","utam_des"],observed=True)["weight"].sum()
    present_edges = edges_d.index.intersection(total_edges.index)
    covered = float(total_edges.loc[present_edges].sum()/total_edges.sum())
    n = len(scope["nodes"])
    graph = np.full((n,n),np.inf)
    for (u,v),row in edges_d.iterrows():
        i,j = scope["index"][u],scope["index"][v]
        if i!=j:
            graph[i,j] = float(row["minutes"])
    np.fill_diagonal(graph,0.0)
    dist = shortest_path(graph,directed=True,unweighted=False)
    return dist,covered

def opportunity_surfaces(scope,dist):
    supported = scope["trips"][scope["trips"]["purpose_group"].isin(PURPOSES)].copy()
    supported["weight_sq"] = supported["weight"]**2
    totals = supported.groupby(["purpose_group","utam_des"],observed=True).agg(
        count=("weight","size"),arrivals=("weight","sum"),sum_sq=("weight_sq","sum")).reset_index()
    totals["ess"] = totals["arrivals"]**2/totals["sum_sq"]
    totals = totals[totals["count"].ge(MIN_DESTINATION_SAMPLE) &
                    totals["ess"].ge(MIN_DESTINATION_ESS)]
    result = {}
    for p in PURPOSES:
        rows = totals[totals["purpose_group"].eq(p) &
                      totals["utam_des"].isin(scope["targets"][p])].copy()
        if set(rows["utam_des"]) != scope["targets"][p]:
            raise RuntimeError(f"{scope['name']}/{p}: unsupported target in benchmark")
        indices = rows["utam_des"].map(scope["index"]).to_numpy(dtype=int)
        weights = rows["arrivals"].to_numpy(dtype=float)
        weights /= weights.sum()
        for cutoff in (45,60):
            scores = (dist[:,indices]<=cutoff).astype(float) @ weights
            result[(p,cutoff)] = scores
    return result

benchmark_rows = []
scopes = (city,region)
for scope in scopes:
    dist,edge_coverage = deterministic_surface(scope)
    scope["duration_edge_weight_coverage"] = edge_coverage
    scope["surface"] = opportunity_surfaces(scope,dist)
    for p in PURPOSES:
        targets = np.array(sorted(scope["index"][u] for u in scope["targets"][p]))
        for cutoff in (45,60):
            access = scope["surface"][(p,cutoff)]
            means = {}
            for sex in SEX_LEVELS:
                mu = scope["mu"][sex].copy()
                mu[targets]=0.
                mu/=mu.sum()
                means[sex] = float(mu @ access)
            # Positive means women have less weighted target opportunity within cutoff.
            gap = means["hombre"]-means["mujer"]
            benchmark_rows.append(dict(scope=scope["name"],purpose_group=p,
                cutoff_minutes=cutoff,male_access=means["hombre"],
                female_access=means["mujer"],male_minus_female_access=gap,
                relative_access_gap=gap/((means["mujer"]+means["hombre"])/2),
                edge_weight_duration_coverage=edge_coverage))
benchmark = pd.DataFrame(benchmark_rows)
if min(scope["duration_edge_weight_coverage"] for scope in scopes)<0.95:
    raise RuntimeError("Insufficient positive-duration edge coverage for benchmark")
a=benchmark.pivot(index=["purpose_group","cutoff_minutes"],
    columns="scope",values="male_minus_female_access").reset_index()
a["region_minus_city_access_gap"] = a["bogota_region"]-a["bogota_dc"]
print("Higher access gap = lower female reachable-opportunity share")
display(a.round(4))


## Shared household bootstrap for the conventional residential benchmark


In [ ]:
# Fixed target UTAMs, pooled duration surfaces and household identifiers.
# Inference here concerns the benchmark only; primary FPT intervals come from Gate 13.
B=500
households = pd.Index(sorted(set(region_persons["household"].astype(str)) | set(region_trips["household"].astype(str))))
house_index = {h:j for j,h in enumerate(households)}
pre = {}
for scope in scopes:
    local={}
    for sex in SEX_LEVELS:
        subset=scope["residents"][scope["residents"]["sex_group"].eq(sex).fillna(False)]
        local[sex]=(
            subset["residence_utam"].map(scope["index"]).to_numpy(dtype=int),
            subset["household"].astype(str).map(house_index).to_numpy(dtype=int),
            subset["weight"].to_numpy(dtype=float))
    pre[scope["name"]]=local

def weighted_masses(scope,multiplier):
    masses={}
    for sex in SEX_LEVELS:
        residence,hh,weight=pre[scope["name"]][sex]
        masses[sex]=np.bincount(residence,weights=weight*multiplier[hh],minlength=len(scope["nodes"]))
    return masses

def access_gap(scope,p,cutoff,masses):
    target = np.array(sorted(scope["index"][u] for u in scope["targets"][p]))
    score = scope["surface"][(p,cutoff)]
    means={}
    for sex in SEX_LEVELS:
        mass=masses[sex].copy()
        mass[target]=0.
        if mass.sum()<=0:
            return np.nan
        means[sex]=float((mass/mass.sum())@score)
    return means["hombre"]-means["mujer"]

rng=np.random.default_rng(202323)
replicates=[]
for b in range(B):
    mult=rng.poisson(1,size=len(households))
    city_mass=weighted_masses(city,mult)
    region_mass=weighted_masses(region,mult)
    for p in PURPOSES:
        for cutoff in (45,60):
            dc=access_gap(city,p,cutoff,city_mass)
            rg=access_gap(region,p,cutoff,region_mass)
            replicates.append(dict(replicate=b+1,purpose_group=p,cutoff_minutes=cutoff,
                district_access_gap=dc,regional_access_gap=rg,region_minus_city_access_gap=rg-dc))
boot=pd.DataFrame(replicates)
ci=boot.groupby(["purpose_group","cutoff_minutes"]).agg(
    ci_lower=("region_minus_city_access_gap",lambda x:x.quantile(0.025)),
    ci_upper=("region_minus_city_access_gap",lambda x:x.quantile(0.975)),
    bootstrap_sd=("region_minus_city_access_gap","std")).reset_index()
conventional=a.merge(ci,on=["purpose_group","cutoff_minutes"],validate="one_to_one")
conventional.to_csv(OUT/"conventional_accessibility_comparison.csv",index=False)
display(conventional.round(4))


## Reduced paired bootstrap: fixed targets versus reselected targets (M10)


In [ ]:
# This diagnostic tests uncertainty from *target selection*. Like Gate 13, it
# holds the validated SCC and full-sample population shrinkage kernel fixed.
# It does reweight group row ESS and residence masses within each resample.
B_TARGET=150
def bootstrap_fpt_scope(scope,mult):
    n=len(scope["nodes"])
    t=scope["trips"]
    hh=t["household"].astype(str).map(house_index).to_numpy(dtype=int)
    count=mult[hh].astype(float)
    w=scope["trip_weight"]*count
    kernels={}
    mus={}
    for sex in SEX_LEVELS:
        mask=t["sex_group"].eq(sex).fillna(False).to_numpy(dtype=bool)
        ii,jj=scope["origin"][mask],scope["destination"][mask]
        ws=w[mask]
        W=np.zeros((n,n))
        np.add.at(W,(ii,jj),ws)
        out=W.sum(axis=1)
        # Frequency-weighted sum of original squared weights (Poisson household counts).
        sq=np.bincount(ii,weights=scope["trip_weight"][mask]**2*count[mask],minlength=n)
        ess=np.divide(out**2,sq,out=np.zeros(n),where=sq>0)
        emp=np.divide(W,out[:,None],out=np.zeros_like(W),where=out[:,None]>0)
        omega=ess/(ess+TAU)
        kernels[sex]=omega[:,None]*emp+(1-omega[:,None])*scope["P0"]
        res,person_hh,person_weight=pre[scope["name"]][sex]
        mass=np.bincount(res,weights=person_weight*mult[person_hh],minlength=n)
        if mass.sum()<=0:
            raise RuntimeError("Bootstrap residential group empty")
        mus[sex]=mass/mass.sum()
    valid=t["purpose_group"].isin(PURPOSES).to_numpy(dtype=bool) & (count>0)
    purposes=t.loc[valid,"purpose_group"].to_numpy()
    destinations=scope["destination"][valid]
    sampling_count=count[valid]
    sampling_weight=w[valid]
    original=scope["trip_weight"][valid]
    results={}
    for p in PURPOSES:
        pm=purposes==p
        rows=np.bincount(destinations[pm],weights=sampling_count[pm],minlength=n)
        arrival=np.bincount(destinations[pm],weights=sampling_weight[pm],minlength=n)
        sq=np.bincount(destinations[pm],weights=original[pm]**2*sampling_count[pm],minlength=n)
        ess=np.divide(arrival**2,sq,out=np.zeros(n),where=sq>0)
        supported=np.flatnonzero((rows>=MIN_DESTINATION_SAMPLE) &
                                 (ess>=MIN_DESTINATION_ESS) & (arrival>0))
        if len(supported)==0:
            raise RuntimeError(f"No bootstrap targets for {scope['name']}/{p}")
        ranked=supported[np.argsort(-arrival[supported],kind="stable")]
        crossing=np.flatnonzero(np.cumsum(arrival[ranked])/arrival[ranked].sum()>=0.5)
        k=int(crossing[0]+1) if len(crossing) else len(ranked)
        new=set(ranked[:k])
        fixed={scope["index"][u] for u in scope["targets"][p]}
        outcomes={}
        for definition,targets in (("fixed",fixed),("reselected",new)):
            target=np.array(sorted(targets))
            outside=np.array([j for j in range(n) if j not in targets])
            fpt={}
            for sex in SEX_LEVELS:
                Q=kernels[sex][np.ix_(outside,outside)]
                h=np.zeros(n)
                h[outside]=np.linalg.solve(np.eye(len(outside))-Q,np.ones(len(outside)))
                mu=mus[sex].copy()
                mu[target]=0.
                if mu.sum()<=0:
                    raise RuntimeError("Bootstrap outside-target mass empty")
                fpt[sex]=float((mu/mu.sum())@h)
            outcomes[definition]=fpt["mujer"]-fpt["hombre"]
        results[p]=dict(fixed=outcomes["fixed"],reselected=outcomes["reselected"],
                        target_jaccard=len(fixed&new)/len(fixed|new))
    return results

rng_target=np.random.default_rng(20232310)
target_reps=[]
for b in range(B_TARGET):
    mult=rng_target.poisson(1,size=len(households))
    city_b=bootstrap_fpt_scope(city,mult)
    region_b=bootstrap_fpt_scope(region,mult)
    for p in PURPOSES:
        target_reps.append({"replicate":b+1,"purpose_group":p,
            "fixed_paired_gap":region_b[p]["fixed"]-city_b[p]["fixed"],
            "reselected_paired_gap":region_b[p]["reselected"]-city_b[p]["reselected"],
            "city_target_jaccard":city_b[p]["target_jaccard"],
            "region_target_jaccard":region_b[p]["target_jaccard"]})
target_boot=pd.DataFrame(target_reps)
target_summary=target_boot.groupby("purpose_group").agg(
    fixed_lo=("fixed_paired_gap",lambda x:x.quantile(.025)),
    fixed_hi=("fixed_paired_gap",lambda x:x.quantile(.975)),
    reselected_lo=("reselected_paired_gap",lambda x:x.quantile(.025)),
    reselected_hi=("reselected_paired_gap",lambda x:x.quantile(.975)),
    median_city_target_jaccard=("city_target_jaccard","median"),
    median_region_target_jaccard=("region_target_jaccard","median")
).reset_index()
target_summary["fixed_interval_width"]=target_summary["fixed_hi"]-target_summary["fixed_lo"]
target_summary["reselected_interval_width"]=target_summary["reselected_hi"]-target_summary["reselected_lo"]
target_summary.to_csv(OUT/"target_reselection_bootstrap_summary.csv",index=False)
target_boot.to_csv(OUT/"target_reselection_bootstrap_replicates.csv",index=False)
display(target_summary.round(4))


## Home-origin kernel check, decision card and persistent outputs


In [ ]:
primary=pd.concat([model(city),model(region)],ignore_index=True)
home=pd.concat([model(city,home_only=True),model(region,home_only=True)],ignore_index=True)
endpoint=primary.pivot(index="purpose_group",columns="scope",values="gap")
home_endpoint=home.pivot(index="purpose_group",columns="scope",values="gap")
decision=pd.DataFrame({
    "purpose_group":list(PURPOSES),
    "primary_paired_fpt_gap":[float(endpoint.loc[p,"bogota_region"]-endpoint.loc[p,"bogota_dc"]) for p in PURPOSES],
    "home_only_paired_fpt_gap":[float(home_endpoint.loc[p,"bogota_region"]-home_endpoint.loc[p,"bogota_dc"]) for p in PURPOSES],
    "home_only_regional_fpt_gap":[float(home_endpoint.loc[p,"bogota_region"]) for p in PURPOSES],
})
decision["retention_ratio"]=decision["home_only_paired_fpt_gap"]/decision["primary_paired_fpt_gap"]
for cutoff in (45,60):
    z=conventional[conventional["cutoff_minutes"].eq(cutoff)].set_index("purpose_group")
    decision[f"access_{cutoff}_paired_gap"] = [float(z.loc[p,"region_minus_city_access_gap"]) for p in PURPOSES]
    decision[f"access_{cutoff}_ci_lower"] = [float(z.loc[p,"ci_lower"]) for p in PURPOSES]
    decision[f"access_{cutoff}_ci_upper"] = [float(z.loc[p,"ci_upper"]) for p in PURPOSES]
decision=decision.merge(target_summary,on="purpose_group",validate="one_to_one")
decision.to_csv(OUT/"one_run_decision_card.csv",index=False)
primary.to_csv(OUT/"replicated_primary_endpoints.csv",index=False)
home.to_csv(OUT/"home_origin_fpt_endpoints.csv",index=False)
pd.DataFrame([support]).to_csv(OUT/"coverage_and_home_support.csv",index=False)
checks={
    "gate11_and_gate12_points_reproduced":bool(max(region_error,city_error)<=1e-8),
    "full_regional_node_set_reproduced":bool(region["nodes"]==sorted(node_set)),
    "benchmark_edge_weight_coverage_at_least_95_percent":bool(min(s["duration_edge_weight_coverage"] for s in scopes)>=.95),
    "all_benchmark_bootstrap_replicates_finite":bool(len(boot)==B*len(PURPOSES)*2 and
        np.isfinite(boot.select_dtypes(include="number").to_numpy()).all()),
    "home_origin_sample_present_for_both_sexes":bool(all(
        int((s["trips"]["sex_group"].eq(sex)&s["trips"]["from_home"]).sum())>0
        for s in scopes for sex in SEX_LEVELS)),
    "decision_card_finite":bool(np.isfinite(decision.select_dtypes(include="number").to_numpy()).all()),
    "target_reselection_bootstrap_complete":bool(len(target_boot)==B_TARGET*len(PURPOSES) and
        np.isfinite(target_boot.select_dtypes(include="number").to_numpy()).all()),
}
checks["gate_23_passed"]=bool(all(checks.values()))
report={"generated_utc":datetime.now(timezone.utc).isoformat(),
    "experiment":"M7 deterministic travel-time benchmark, M1 home-origin kernel, M10 target-reselection bootstrap",
    "primary_estimate":"female-minus-male mean FPT and paired region-minus-district difference",
    "benchmark":"male-minus-female share of pooled weighted core-50 destination arrivals reachable within 45/60 min",
    "bootstrap":"500 shared Poisson(1) households for benchmark residential means; 150 shared households for fixed-versus-reselected FPT targets; neither replaces Gate 13 simultaneous max-t intervals",
    "limitations":"revealed-flow travel time proxy; home-origin comparison and primary FPT gap cannot inherit benchmark CIs",
    "support":support,"gate":checks,
    "decision_card":json.loads(decision.to_json(orient="records"))}
(OUT/"gate_23.json").write_text(json.dumps(report,ensure_ascii=False,indent=2,allow_nan=False),encoding="utf-8")
display(decision.round(4))
display(pd.Series(checks))
if not checks["gate_23_passed"]:
    raise RuntimeError("Gate 23 audit failed; no scientific decision should be made")
persistent=DRIVE_BASE/"results/gate23"
persistent.mkdir(parents=True,exist_ok=True)
for file in OUT.iterdir():
    if file.is_file():
        shutil.copy2(file,persistent/file.name)
print("Gate 23 complete:",persistent)
